# BO Forge maximisation LogEI campaign

This notebook demonstrates the v0.2 `CampaignSession` workflow one experiment at a time.

Each cycle requests one candidate, appends it as `status=suggested`, simulates the lab result, marks that same row as `status=observed`, reloads the CSV log through the session, and only then requests the next candidate.


## 1. Setup

The example config uses a sequential campaign. The session object owns the config, log path, and current campaign DataFrame.


In [ ]:
from pathlib import Path
import os
import shutil
import sys

import numpy as np
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
mpl_cache = PROJECT_ROOT / ".matplotlib-cache"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from bo_forge.session import CampaignSession


In [ ]:
config_path = PROJECT_ROOT / "configs" / "simple_2d.yaml"
seed_log_path = PROJECT_ROOT / "examples" / "simple_2d_campaign_log.csv"
working_log_path = PROJECT_ROOT / "examples" / "simple_2d_working_log.csv"
latest_suggestion_path = PROJECT_ROOT / "examples" / "latest_suggestions.csv"

shutil.copyfile(seed_log_path, working_log_path)

campaign = CampaignSession.from_files(config_path=config_path, log_path=working_log_path)
one_at_a_time_batch_size = 1


In [ ]:
def simulated_activity(row) -> float:
    ratio = float(row["precursor_ratio"])
    temperature = float(row["annealing_temperature"])
    smooth_peak = 2.2 - 4.0 * (ratio - 0.62) ** 2
    smooth_peak -= ((temperature - 710.0) / 170.0) ** 2
    small_wave = 0.04 * np.sin(10.0 * ratio)
    small_wave += 0.03 * np.cos(temperature / 55.0)
    return float(smooth_peak + small_wave)


def current_log():
    campaign.reload()
    campaign.validate()
    return campaign.df


def request_one_candidate():
    suggestion = campaign.suggest_next(batch_size=one_at_a_time_batch_size)
    suggestion.to_csv(latest_suggestion_path, index=False)
    campaign.append_suggestions(suggestion)
    return suggestion


def enter_simulated_result(suggestion):
    row = suggestion.iloc[0]
    result = simulated_activity(row)
    campaign.mark_observed(str(row["row_id"]), result)
    return result


## 2. Load and summarise the current campaign

The campaign starts from two manually observed experiments. The remaining initial-design points will be suggested one at a time using Sobol.


In [ ]:
campaign.validate()
display(campaign.summary())
campaign.df


## 3. Request one initial candidate

In [ ]:
suggestion = request_one_candidate()
suggestion

## 4. Enter the result for that one experiment

In a real campaign, replace `simulated_activity(...)` with the measured lab result, then call `campaign.mark_observed(...)` for the same `row_id`.


In [ ]:
result = enter_simulated_result(suggestion)
df = current_log()
print("Entered result:", result)
df.tail(3)

## 5. Repeat the same one-experiment cycle

Each cell below represents one lab iteration: suggest one candidate, run one experiment, enter one result, reload the log.

In [ ]:
suggestion = request_one_candidate()
result = enter_simulated_result(suggestion)
df = current_log()
display(suggestion)
print("Entered result:", result)
df.loc[df["row_id"] == str(suggestion.loc[0, "row_id"])]

In [ ]:
suggestion = request_one_candidate()
result = enter_simulated_result(suggestion)
df = current_log()
display(suggestion)
print("Entered result:", result)
df.loc[df["row_id"] == str(suggestion.loc[0, "row_id"])]

In [ ]:
suggestion = request_one_candidate()
result = enter_simulated_result(suggestion)
df = current_log()
display(suggestion)
print("Entered result:", result)
df.loc[df["row_id"] == str(suggestion.loc[0, "row_id"])]

In [ ]:
suggestion = request_one_candidate()
result = enter_simulated_result(suggestion)
df = current_log()
display(suggestion)
print("Entered result:", result)
df.loc[df["row_id"] == str(suggestion.loc[0, "row_id"])]

In [ ]:
suggestion = request_one_candidate()
result = enter_simulated_result(suggestion)
df = current_log()
display(suggestion)
print("Entered result:", result)
df.loc[df["row_id"] == str(suggestion.loc[0, "row_id"])]

## 6. Now the initial design is complete

Once the number of observed rows reaches `initial_design_size`, the next session suggestion fits the GP and uses LogEI because this notebook requests one candidate at a time.


In [ ]:
len(campaign.observed_data()), campaign.config.bo.initial_design_size

In [ ]:
suggestion = request_one_candidate()
suggestion

In [ ]:
result = enter_simulated_result(suggestion)
df = current_log()
print("Entered result:", result)
df.loc[df["row_id"] == str(suggestion.loc[0, "row_id"])]

## 7. Diagnostics

In [ ]:
campaign.plot_progress();

In [ ]:
campaign.plot_diagnostics();